In [154]:
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings

from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI
from langchain_community.document_loaders import UnstructuredMarkdownLoader

from dotenv import load_dotenv,find_dotenv
_ = load_dotenv(find_dotenv(),verbose=True)

In [155]:
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to /home/dev/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [252]:
import os

from langchain_community.document_loaders import UnstructuredMarkdownLoader
from langchain_core.documents import Document


def get_chunks(path):
    basename = os.path.basename(path)
    project_name = basename[:-4]
    loader = UnstructuredMarkdownLoader(path)
    pages = loader.load_and_split()
    print(f"path: {path}, pages: {len(pages)}")
    # 文档切分
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=300,
        chunk_overlap=100,
        length_function=len,
        add_start_index=True,
    )
    results = []
    for page in pages:
        source = page.metadata["source"]
        page_content = page.page_content
        texts = text_splitter.create_documents([page_content])
        for i, t in enumerate(texts):
            texts[i].metadata["doc_source"] = source
            texts[i].metadata["project_name"] = project_name
        results += texts.copy()
    return results


chunks = []
for filename in os.listdir("./rag"):
    if filename.endswith("txt") == False:
        continue
    path = os.path.join("./rag", filename)
    chunks += get_chunks(path)

print(len(chunks))
# print(chunks[0])

path: rag/news_business_04.txt, pages: 2
path: rag/news_business_03.txt, pages: 2
path: rag/news_business_06.txt, pages: 1
path: rag/news_business_01.txt, pages: 2
path: rag/news_business_02.txt, pages: 3
path: rag/news_business_05.txt, pages: 1
151


In [253]:
# 灌库
embeddings = OllamaEmbeddings(model="bge-m3")
db = FAISS.from_documents(chunks, embeddings)
retriever = db.as_retriever(search_kwargs={"k": 30}) 

In [254]:
import os
os.environ['OPENAI_API_KEY'] = "sk-xxx"
os.environ['OPENAI_BASE_URL'] = "https://llm_provider_url"

model_name = "deepseek-r1"

from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

from langchain_core.prompts import (
    ChatPromptTemplate,
    HumanMessagePromptTemplate,
    SystemMessagePromptTemplate,
)

# Prompt模板
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)
llm = ChatOpenAI(model=model_name)

query = "FPT对Blueward的战略投资计划取消了吗"
retriever_results = retriever.invoke(query)

In [255]:
doc_v1 = str(retriever_results)
doc_v2 = ""

for doc in retriever_results:
    doc_v2 += "相关信息: \n" + doc.page_content + "\n\n"


In [256]:
c = """
投资状态相关的问题应该参考如下信息。包含投资状态，对应投资历史文档可通过source关联。
项目: news_business_01, 状态: "已取消", 历史文档: metadata={"source“:'rag/news_business_01.txt'}
项目: news_business_03, 状态: "正常", 历史文档: metadata={"source“:'rag/news_business_03.txt'}
需要注意的是文档一般不会提供投资状态信息。

"""

In [257]:
p1 = template.replace("{context}", c + doc_v1).replace("{question}", query)
p2 = template.replace("{context}", c + doc_v2).replace("{question}", query)

In [258]:
p1

'Answer the question based only on the following context:\n\n投资状态相关的问题应该参考如下信息。包含投资状态，对应投资历史文档可通过source关联。\n项目: news_business_01, 状态: "已取消", 历史文档: metadata={"source“:\'rag/news_business_01.txt\'}\n项目: news_business_03, 状态: "正常", 历史文档: metadata={"source“:\'rag/news_business_03.txt\'}\n需要注意的是文档一般不会提供投资状态信息。\n\n[Document(id=\'c7851b62-fc0e-4de5-b65b-3744b23c669c\', metadata={\'start_index\': 0, \'doc_source\': \'rag/news_business_01.txt\', \'project_name\': \'news_business_01\'}, page_content=\'FPT Strengthens South Korea Footprint with Strategic Partnership and Investment in Blueward\'), Document(id=\'e4ffee8c-3043-457a-bca7-ff7b265101fe\', metadata={\'start_index\': 294, \'doc_source\': \'rag/news_business_01.txt\', \'project_name\': \'news_business_01\'}, page_content=\'signing of a Strategic Investment Agreement and a Master Service Agreement. Through its subsidiary in South Korea, FPT will make a strategic investment to secure up to a 10% equity stake in Blueward. The investment is p

In [259]:
llm.invoke(p1)

AIMessage(content='根据提供的上下文信息，**FPT对Blueward的战略投资计划已被取消**。具体依据如下：\n\n1. 在项目状态列表中明确标注：  \n   - **项目: news_business_01, 状态: "已取消"**  \n   - 该项目对应的内容正是FPT与Blueward的战略投资合作（如文档中的"Strategic Investment Agreement"等描述）。\n\n2. 尽管相关文档（`rag/news_business_01.txt`）详细描述了投资计划（如“2028年前完成投资”“获得10%股权”等），但**项目状态独立于文档内容**，且状态说明已明确标记为“已取消”。\n\n3. 上下文特别强调：**“文档一般不会提供投资状态信息”**，因此需以项目状态列表为准。\n\n**结论**：该投资计划当前状态为**已取消**。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 449, 'prompt_tokens': 3371, 'total_tokens': 3820, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 264, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'deepseek-r1', 'system_fingerprint': None, 'id': 'chatcmpl-ca59d906-bd0b-4b98-85c8-768492ffb47f', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019afd87-f33d-7b41-87ff-2c2f7978b7ab-0', usage_metadata={'input_tokens': 3371,

In [260]:
llm.invoke(p2)

AIMessage(content='根据提供的信息，**FPT对Blueward的战略投资计划没有取消**。\n\n关键依据：\n1. 上下文明确提到双方已签署"战略投资协议"（Strategic Investment Agreement）\n2. 具体说明"FPT将通过韩国子公司进行战略投资，以获取Blueward最多10%的股权"\n3. 明确时间节点："投资计划在2028年Blueward IPO前完成"\n4. 双方合作目标清晰："旨在加强SAP ERP交付能力，深化咨询专长，提升在韩国市场的竞争力"\n\n虽然投资状态参考信息中包含一个状态为"已取消"的项目（news_business_01），但当前查询涉及的投资计划在上下文中显示出正常推进的积极信号，且无任何提及取消或变更的信息。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 379, 'prompt_tokens': 1544, 'total_tokens': 1923, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 218, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'deepseek-r1', 'system_fingerprint': None, 'id': 'chatcmpl-b27b5e61-0851-4075-b898-ce16fdf6cc55', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019afd88-23fc-71d1-a8b6-18ec4b0c3326-0', usage_metadata={'input_tokens': 1544, 'output_tokens': 379, 'total_tokens': 1923, '